# Deliverable 2 - INF201 (Tasks 1 and 2)
**Group members:** Rishanth Jeevahen. Djordje Pesut
**GitHub repository:** _(legg til lenken her bro:)_

In [1]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

# generate_data_for_d2.py creates "d2_data"; change to "data" if you renamed it
DATA_DIR = Path("d2_data") if Path("d2_data").exists() else Path("data")

## Task 1.1: Robust file reader

In [2]:
def read_log_file(file_path: Path) -> tuple[str, str]:
    """Read a UTF-8 or ISO-8859-1 file. Returns (file_text_content, encoding_used)."""
    file_path = Path(file_path)
    try:
        content = file_path.read_text(encoding="utf-8")
        return content, "utf-8"
    except UnicodeDecodeError:
        content = file_path.read_text(encoding="ISO-8859-1")
        return content, "ISO-8859-1"

## Task 1.2: Log parser with regular expressions
Only lines in the format `[timestamp] [register_id] item=... price=... CUR (STATUS)` match; noise lines are ignored.

In [3]:
LINE_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>-?\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$",
    re.MULTILINE,
)

def parse_log_file(content: str) -> list[dict]:
    """Return one dictionary per valid sales line; price is converted to float."""
    records = []
    for match in LINE_PATTERN.finditer(content):
        record = match.groupdict()
        record["price"] = float(record["price"])
        records.append(record)
    return records

In [4]:
sample = """--- PRINTER PAPER LOW WARNING ---
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
=== NETWORK SYNC OK ===
[2026-10-15 08:31:10] [POS_02] item=Muffin price=999.00 NOK (REFUND)
"""
parse_log_file(sample)

[{'timestamp': '2026-10-15 08:30:00',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.5,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 08:31:10',
  'register_id': 'POS_02',
  'item': 'Muffin',
  'price': 999.0,
  'currency': 'NOK',
  'status': 'REFUND'}]

## Task 2.1: Recursive directory search (no glob / rglob / os.walk)

In [5]:
def find_log_files(directory, extension: str = ".log") -> list[Path]:
    """Recursively find all files with the given extension."""
    directory = Path(directory)
    found: list[Path] = []
    for item in directory.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    return found

## Task 2.2: Recursive search for active registers in the JSON config

In [6]:
def get_active_register_ids(node) -> set[str]:
    """Return ids of all registers with "active": true, at any nesting depth."""
    active: set[str] = set()
    if isinstance(node, dict):
        if node.get("active") is True and isinstance(node.get("id"), str):
            active.add(node["id"])
        for value in node.values():
            active |= get_active_register_ids(value)
    elif isinstance(node, list):
        for element in node:
            active |= get_active_register_ids(element)
    return active

In [7]:
example = {"company": "Campus Coffee", "regions": [{"region_name": "Main Campus", "stores": [
    {"name": "Library Cafe", "registers": [{"id": "POS_01", "active": True}, {"id": "POS_02", "active": False}]}]}]}
get_active_register_ids(example)

{'POS_01'}

## Test on the real data

In [8]:
log_files = find_log_files(DATA_DIR)
config = json.loads((DATA_DIR / "store_config.json").read_text(encoding="utf-8"))
active_ids = get_active_register_ids(config)

all_records = []
for path in log_files:
    text, encoding = read_log_file(path)
    records = parse_log_file(text)
    all_records.extend(records)
    print(f"{path.relative_to(DATA_DIR)}: {encoding}, {len(records)} valid lines")

print("Log files:", len(log_files), "| records:", len(all_records), "| active:", sorted(active_ids))
pd.DataFrame(all_records).head()

2026\library_cafe\POS_01\register_sales.log: utf-8, 80 valid lines
2026\library_cafe\POS_02\register_sales.log: utf-8, 80 valid lines
2026\library_cafe\POS_03\register_sales.log: utf-8, 80 valid lines
2026\science_building\POS_01\register_sales.log: utf-8, 80 valid lines
2026\science_building\POS_02\register_sales.log: utf-8, 80 valid lines
2026\science_building\POS_03\register_sales.log: utf-8, 80 valid lines
2026\student_union\POS_01\register_sales.log: utf-8, 80 valid lines
2026\student_union\POS_02\register_sales.log: utf-8, 80 valid lines
2026\student_union\POS_03\register_sales.log: utf-8, 80 valid lines
Log files: 9 | records: 720 | active: ['POS_01', 'POS_02', 'POS_03']


,timestamp,register_id,item,price,currency,status
0,2026-10-15 07:01:35,POS_01,Cappuccino,4.32,USD,SUCCESS
1,2026-10-15 07:03:53,POS_01,Latte,4.44,EUR,SUCCESS
2,2026-10-15 07:06:34,POS_01,Muffin,3.44,USD,CANCELLED
3,2026-10-15 07:08:32,POS_01,Muffin,999.00,USD,REFUND
4,2026-10-15 07:08:57,POS_01,Sandwich,7.85,USD,SUCCESS


## Discussion
- The reader tries UTF-8 first and falls back to ISO-8859-1 (which decodes any byte), so the encoding is detected, not hard-coded. ASCII-only files are valid in both, so they are reported as UTF-8.
- The regex is anchored to the whole line, so printer/network/debug noise never matches.
- Both recursive functions call themselves on sub-structures (subfolders, nested dicts/lists).